In [1]:
import folium, pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon

esri = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}"

spacex_df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv")
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()[['Launch Site', 'Lat', 'Long']]
print(launch_sites_df)

map1 = folium.Map(location=[29.56, -95.08], zoom_start=4, tiles=esri, attr="Esri")
for i, row in launch_sites_df.iterrows():
    c = [row['Lat'], row['Long']]
    map1.add_child(folium.Circle(c, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(row['Launch Site'])))
    map1.add_child(folium.map.Marker(c, icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row['Launch Site'])))
map1.save(r"C:\Users\DELL\Downloads\map1.html")

    Launch Site        Lat        Long
0   CCAFS LC-40  28.562302  -80.577356
1  CCAFS SLC-40  28.563197  -80.576820
2    KSC LC-39A  28.573255  -80.646895
3   VAFB SLC-4E  34.632834 -120.610745


In [2]:
spacex_df['marker_color'] = spacex_df['class'].apply(lambda c: 'green' if c == 1 else 'red')
map2 = folium.Map(location=[29.56, -95.08], zoom_start=4, tiles=esri, attr="Esri")
cluster = MarkerCluster()
map2.add_child(cluster)
for i, r in spacex_df.iterrows():
    cluster.add_child(folium.Marker([r['Lat'], r['Long']],
        icon=folium.Icon(color='white', icon_color=r['marker_color']),
        popup=r['Launch Site'] + (' - success' if r['class'] == 1 else ' - failure')))
map2.save(r"C:\Users\DELL\Downloads\map2.html")

In [3]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    a = sin((lat2-lat1)/2)**2 + cos(lat1)*cos(lat2)*sin((lon2-lon1)/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

site = launch_sites_df[launch_sites_df['Launch Site'].str.contains('SLC-40')].iloc[0]
site_lat, site_lon = site['Lat'], site['Long']

map3 = folium.Map(location=[site_lat, site_lon], zoom_start=12, tiles=esri, attr="Esri")
map3.add_child(folium.Circle([site_lat, site_lon], radius=300, color='#d35400', fill=True).add_child(folium.Popup(site['Launch Site'])))

points = {'Coastline': (28.56367, -80.57163), 'Railway': (28.57206, -80.58529),
          'Highway': (28.56452, -80.57105), 'City': (28.39772, -80.60670)}
for name, (lat, lon) in points.items():
    d = calculate_distance(site_lat, site_lon, lat, lon)
    print(f"{name}: {d:.2f} km")
    map3.add_child(folium.Marker([lat, lon], icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % f"{name}: {d:.2f} KM")))
    map3.add_child(folium.PolyLine([[site_lat, site_lon], [lat, lon]], weight=1))
map3.save(r"C:\Users\DELL\Downloads\map3.html")

Coastline: 0.51 km
Railway: 1.29 km
Highway: 0.58 km
City: 18.64 km


In [4]:
print(launch_sites_df)
print(spacex_df.groupby('Launch Site')['class'].agg(['count', 'sum', 'mean']))

    Launch Site        Lat        Long
0   CCAFS LC-40  28.562302  -80.577356
1  CCAFS SLC-40  28.563197  -80.576820
2    KSC LC-39A  28.573255  -80.646895
3   VAFB SLC-4E  34.632834 -120.610745
              count  sum      mean
Launch Site                       
CCAFS LC-40      26    7  0.269231
CCAFS SLC-40      7    3  0.428571
KSC LC-39A       13   10  0.769231
VAFB SLC-4E      10    4  0.400000
